In [1]:
import pandas as pd
import numpy as np
import os

## Part 1

The source of this data comes from NASA's NeoWs (Near Earth Object Web Service). When requesting data from the API, the end data can be a maximum of seven days after the start date. The raw data contains lots of information about asteriods that are considered Near Earth Objects (NEOs). In this project, only seven variable from the raw data are considered, these seven variables include: Approach Date, Miss Distance, Relative Velocity, Object Id (primary key), Absolute Magnitude, Estimated Diameter, and Potentially Hazardous Asteriod flag.

Every approach date contians approach data from an object on that day. Because of this reason, the data is read in as two seperate DataFrames that are merged on approach date. Approach date is used as the key for this merge, however, moving forward, Object Id is used as the primary key as it is more practical.

The first DataFrame created contains the Approach Date, Miss Distance, and Relative Velocity for every NEO observed on that day. The second DataFrame contains the Approach Date, Object Id, Absolute Magnitude, Estimated Diameter, and Potentially Hazardous Asteriod flag. The final dataset containing all NEO observations is titled neos_df.

The neos_df contains 7 columns, 480 rows, and no missing values. The full structure of the DataFrame is as follows:
<pre>
 #   Column                             Non-Null Count  Dtype  
 0   Approach Date                      480 non-null    datetime64[us]
 1   miss_distance                      480 non-null    float64       
 2   relative_velocity                  480 non-null    float64       
 3   object_id                          480 non-null    str           
 4   absolute_magnitude_h               480 non-null    float64       
 5   estimated_diameter                 480 non-null    float64       
 6   is_potentially_hazardous_asteroid  480 non-null    bool
 <pre>

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

from read_data_from_api import request_data, get_close_approach_data, get_object_data, api_data_to_dataframe

## Raw Data Exploration

In [3]:
data = request_data(start_date="2026-10-21", end_date="2026-10-28")

print(data.keys())

dict_keys(['element_count', 'links', 'near_earth_objects'])


In [4]:
neos = data['near_earth_objects']
neos.keys()

dict_keys(['2026-10-28', '2026-10-27', '2026-10-22', '2026-10-21', '2026-10-26', '2026-10-25', '2026-10-24', '2026-10-23'])

In [5]:
neos['2026-10-28'][0]

{'absolute_magnitude_h': 21.85,
 'close_approach_data': [{'close_approach_date': '2026-10-28',
   'close_approach_date_full': '2026-Oct-28 16:55',
   'epoch_date_close_approach': 1793206500000,
   'miss_distance': {'astronomical': '0.1530448084',
    'kilometers': '22895177.351198108',
    'lunar': '59.5344304676',
    'miles': '14226403.5189375704'},
   'orbiting_body': 'Earth',
   'relative_velocity': {'kilometers_per_hour': '20345.8248160687',
    'kilometers_per_second': '5.6516180045',
    'miles_per_hour': '12642.1043141844'}}],
 'estimated_diameter': {'feet': {'estimated_diameter_max': 831.8117101206,
   'estimated_diameter_min': 371.997505662},
  'kilometers': {'estimated_diameter_max': 0.2535362011,
   'estimated_diameter_min': 0.1133848361},
  'meters': {'estimated_diameter_max': 253.5362011316,
   'estimated_diameter_min': 113.3848360975},
  'miles': {'estimated_diameter_max': 0.1575400428,
   'estimated_diameter_min': 0.070454049}},
 'id': '2518810',
 'is_potentially_hazard

In [6]:
neos['2026-10-28'][0]['close_approach_data']

[{'close_approach_date': '2026-10-28',
  'close_approach_date_full': '2026-Oct-28 16:55',
  'epoch_date_close_approach': 1793206500000,
  'miss_distance': {'astronomical': '0.1530448084',
   'kilometers': '22895177.351198108',
   'lunar': '59.5344304676',
   'miles': '14226403.5189375704'},
  'orbiting_body': 'Earth',
  'relative_velocity': {'kilometers_per_hour': '20345.8248160687',
   'kilometers_per_second': '5.6516180045',
   'miles_per_hour': '12642.1043141844'}}]

In [7]:
approach_dates = [app_date for app_date in neos]

close_approach_df = get_close_approach_data(neos=neos, approach_dates=approach_dates)
close_approach_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Approach Date      60 non-null     str    
 1   miss_distance      60 non-null     float64
 2   relative_velocity  60 non-null     float64
dtypes: float64(2), str(1)
memory usage: 1.5 KB


In [8]:
object_data_df = get_object_data(neos=neos, approach_dates=approach_dates)
object_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Approach Date                      60 non-null     str    
 1   object_id                          60 non-null     str    
 2   absolute_magnitude_h               60 non-null     float64
 3   estimated_diameter                 60 non-null     float64
 4   is_potentially_hazardous_asteroid  60 non-null     bool   
dtypes: bool(1), float64(2), str(2)
memory usage: 2.1 KB


## Request and save data

In [9]:
neos_df = api_data_to_dataframe(api_data=data)
display(neos_df.head())

,Approach Date,miss_distance,relative_velocity,object_id,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid
210,2026-10-26,6.642528e+07,67073.229700,2488509,20.61,0.200704,False
282,2026-10-26,4.635502e+07,32270.110647,2488509,20.61,0.200704,False
273,2026-10-26,4.509338e+07,73474.264537,2488509,20.61,0.200704,False
264,2026-10-26,2.116466e+07,12565.295839,2488509,20.61,0.200704,False
246,2026-10-26,6.486772e+07,71372.128055,2488509,20.61,0.200704,False


In [10]:
neos_df.info()

<class 'pandas.DataFrame'>
Index: 480 entries, 210 to 340
Data columns (total 7 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Approach Date                      480 non-null    datetime64[us]
 1   miss_distance                      480 non-null    float64       
 2   relative_velocity                  480 non-null    float64       
 3   object_id                          480 non-null    str           
 4   absolute_magnitude_h               480 non-null    float64       
 5   estimated_diameter                 480 non-null    float64       
 6   is_potentially_hazardous_asteroid  480 non-null    bool          
dtypes: bool(1), datetime64[us](1), float64(4), str(1)
memory usage: 26.7 KB


In [11]:
csv_fpath = os.path.join("..", "data", "raw", "neo_data.csv")
neos_df.to_csv(csv_fpath, index=False)